# Build knowledge graph


In [ ]:
%env AWS_PROFILE=kmnlp

In [ ]:
import sys

sys.path.append("../src")

In [ ]:
import json
import re
from datetime import UTC, datetime
from typing import Any

import s3fs  # pyright: ignore[reportMissingTypeStubs]
from pydantic import BaseModel, Field
from rich import print as rprint

from llm_agent.utils.knowledge_graph_construction.graph_construction import build_graph, save_graph
from llm_agent.utils.knowledge_graph_construction.metadata_parsing import (
    Dataset,
    Dimension,
    Variable,
    parse,
)


In [ ]:
# If True, create only a single global "canonical" instance of each dim without
# the noisy long_name and units metadata fields.
# If False, create an instance for each occurence of a dim i.e. there will be
# a unique time dim for each dataset. These will also include noisy metadata
# fields.
USE_CANONICAL_DIMS = True

## Load metadata


### Load metadata for CDR datasets


In [ ]:
s3 = s3fs.S3FileSystem()
with s3.open("s3://data-c6c22a2e42294c11b52ee7f0c792c071/ds_metadata.json", "r") as f:  # pyright: ignore[reportUnknownMemberType]
    ds_metadata = json.load(f)

---


## Custom overrides


### Corrected values for `cdr_variable`


For Edge cases where `cdr_variable` does not match name of actual variable.


Compare `cdr_variable`, variable name in data, and `standard_name`:


In [ ]:
# for ds_name, groups in ds_metadata.items():
#     for group_name, group_data in groups.items():
#         if group_data["uri"] is None:
#             continue
#         if "cdr_variable" not in group_data["zarr_metadata"][".zattrs"]:
#             continue
#         print(ds_name, group_name)
#         print(group_data["zarr_metadata"][".zattrs"]["cdr_variable"])
#         print([
#             (k.split("/")[0], f'std_name: {v.get("standard_name")}')
#             for k, v in group_data["zarr_metadata"].items()
#             if (
#                 k.endswith("/.zattrs")
#                 and k.split("/")[0] not in {"lat", "lon", "time", "latitude", "longitude", "nv", "bounds", "crs", "wavelength"}
#                 and not any(v_ in k for v_ in {"UNC", "QA", "_bounds", "_bnds", "_qc", "_Band_Width"})
#             )
#         ])
#         print("---")

In [ ]:
custom_cdr_var_mappings = {
    "noaa-cdr-aerosol-optical-thickness-pds": {
        "daily": "aot1",
        "monthly": "aot1",
    },
    "noaa-cdr-precip-gpcp-daily-pds": {
        "daily": "precip",
    },
    "noaa-cdr-precip-gpcp-monthly-pds": {
        "monthly": "precip",
    },
    "noaa-cdr-precip-cmorph-pds": {
        "daily": "cmorph",
    },
}

### Alternate dim names


In [ ]:
dim_alt_names = {
    "latitude": "lat",
    "longitude": "lon",
    "radiation_wavelength": "wavelength",
}

### Fixed landing page links


In [ ]:
metadata_link_overrides = {
    "noaa-cdr-precip-gpcp-daily-pds": {"daily": "gov.noaa.ncdc:C00999"},
}

## Parse metadata


In [ ]:
def _get_variable_field_names(
    dataset_name: str, dataset_group: str, collection_metadata: dict[str, Any]
) -> list[str]:
    try:
        cdr_variable = custom_cdr_var_mappings[dataset_name][dataset_group]
    except KeyError:
        cdr_variable = collection_metadata["cdr_variable"]
    return [var_name.strip() for var_name in cdr_variable.split(",")]


def _get_metadata_link(
    dataset_name: str, dataset_group: str, collection_metadata: dict[str, Any]
) -> str | None:
    if metadata_link_overrides:
        try:
            return metadata_link_overrides[dataset_name][dataset_group]
        except KeyError:
            pass
    metadata_link = collection_metadata.get("metadata_link")

    # if of the form: gov.noaa.ncdc:C01678
    if metadata_link and re.match(r"gov\.noaa.+:.+", metadata_link):
        metadata_link = (
            f"https://www.ncei.noaa.gov/access/metadata/landing-page/bin/iso?id={metadata_link}"
        )

    return metadata_link


In [ ]:
ds_metadata.keys()

In [ ]:
# rprint(
#     parse(
#         "noaa-cdr-aerosol-optical-thickness-pds",
#         "daily",
#         ds_metadata["noaa-cdr-aerosol-optical-thickness-pds"]["daily"],
#         get_variable_field_names_func=_get_variable_field_names,
#         get_metadata_link_func=_get_metadata_link,
#     )
# )

In [ ]:
datasets: list[Dataset] = []
for ds_name, groups in ds_metadata.items():
    for group_name, group_data in groups.items():
        if group_data["uri"]["parquet"] is None:
            continue
        try:
            dataset = parse(
                ds_name,
                group_name,
                group_data,
                get_variable_field_names_func=_get_variable_field_names,
                get_metadata_link_func=_get_metadata_link,
            )
            datasets.append(dataset)
        except KeyError:
            print("Failed:", ds_name, group_name)  # noqa: T201
            continue


---


## Populate graph


In [ ]:
g = build_graph(
    datasets,
    canonical_dims=True,
    media_type="application/kerchunk-parquet",
)

## Save


In [ ]:
out_path = "../../src/llm_agent/data/dataset-kg-cdr-generated.ttl"
save_graph(g, out_path)